# House Price Prediction Analysis

This notebook provides a step-by-step analysis of house price prediction using the California Housing dataset.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

## 1. Load and Explore Data

In [ ]:
# Load California housing dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target * 100000  # Convert to actual dollars

print("Dataset shape:", df.shape)
print("\nFeatures:", list(df.columns))
print("\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics
print("Dataset Info:")
print(df.info())
print("\n\nBasic Statistics:")
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('California Housing Dataset - Exploratory Analysis', fontsize=16)

# Price distribution
axes[0, 0].hist(df['Price'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('House Price Distribution')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Median income vs Price
axes[0, 1].scatter(df['MedInc'], df['Price'], alpha=0.5, s=10)
axes[0, 1].set_title('Median Income vs House Price')
axes[0, 1].set_xlabel('Median Income ($100k)')
axes[0, 1].set_ylabel('House Price ($)')

# House age vs Price
axes[0, 2].scatter(df['HouseAge'], df['Price'], alpha=0.5, s=10)
axes[0, 2].set_title('House Age vs House Price')
axes[0, 2].set_xlabel('House Age (years)')
axes[0, 2].set_ylabel('House Price ($)')

# Average rooms vs Price
axes[1, 0].scatter(df['AveRooms'], df['Price'], alpha=0.5, s=10)
axes[1, 0].set_title('Average Rooms vs House Price')
axes[1, 0].set_xlabel('Average Rooms')
axes[1, 0].set_ylabel('House Price ($)')

# Population vs Price
axes[1, 1].scatter(df['Population'], df['Price'], alpha=0.5, s=10)
axes[1, 1].set_title('Population vs House Price')
axes[1, 1].set_xlabel('Population')
axes[1, 1].set_ylabel('House Price ($)')

# Correlation heatmap
corr_matrix = df.corr()
im = axes[1, 2].imshow(corr_matrix, cmap='coolwarm', aspect='auto')
axes[1, 2].set_title('Feature Correlation Matrix')
axes[1, 2].set_xticks(range(len(corr_matrix.columns)))
axes[1, 2].set_yticks(range(len(corr_matrix.columns)))
axes[1, 2].set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
axes[1, 2].set_yticklabels(corr_matrix.columns)
plt.colorbar(im, ax=axes[1, 2])

plt.tight_layout()
plt.show()

## 3. Prepare Data for Modeling

In [ ]:
# Split features and target
X = df.drop('Price', axis=1)
y = df['Price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## 4. Train and Evaluate Models

In [ ]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'predictions': y_pred
    }
    
    print(f"  RMSE: ${rmse:,.2f}")
    print(f"  MAE: ${mae:,.2f}")
    print(f"  R²: {r2:.4f}")
    print()

## 5. Compare Model Performance

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'RMSE': [results[m]['rmse'] for m in results.keys()],
    'MAE': [results[m]['mae'] for m in results.keys()],
    'R²': [results[m]['r2'] for m in results.keys()]
}).sort_values('RMSE')

print("Model Performance Comparison:")
comparison_df

In [ ]:
# Visualize model comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# RMSE comparison
ax1.barh(comparison_df['Model'], comparison_df['RMSE'])
ax1.set_xlabel('RMSE ($)')
ax1.set_title('Model RMSE Comparison')
ax1.invert_yaxis()

# R² comparison
ax2.barh(comparison_df['Model'], comparison_df['R²'])
ax2.set_xlabel('R² Score')
ax2.set_title('Model R² Score Comparison')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.show()

print("Feature Importance Rankings:")
print(feature_importance.sort_values('importance', ascending=False))

## 7. Residual Analysis

In [ ]:
# Get best model predictions
best_model_name = comparison_df.iloc[0]['Model']
best_predictions = results[best_model_name]['predictions']

# Calculate residuals
residuals = y_test - best_predictions

# Create residual analysis plots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Residual plot
axes[0, 0].scatter(best_predictions, residuals, alpha=0.5)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Price ($)')
axes[0, 0].set_ylabel('Residuals ($)')
axes[0, 0].set_title('Residual Plot')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')

# Histogram of residuals
axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals ($)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Residual Distribution')

# Actual vs Predicted
axes[1, 1].scatter(y_test, best_predictions, alpha=0.5)
axes[1, 1].plot([y_test.min(), y_test.max()], 
                [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1, 1].set_xlabel('Actual Price ($)')
axes[1, 1].set_ylabel('Predicted Price ($)')
axes[1, 1].set_title('Actual vs Predicted Prices')

plt.tight_layout()
plt.show()

## 8. Make Predictions

In [ ]:
# Example prediction
example_house = {
    'MedInc': 4.0,      # Median income in $100k
    'HouseAge': 20,     # House age in years
    'AveRooms': 6,      # Average number of rooms
    'AveBedrms': 1,     # Average number of bedrooms
    'Population': 1000, # Population
    'AveOccup': 3,      # Average occupancy
    'Latitude': 34.0,   # Latitude
    'Longitude': -118.0 # Longitude
}

# Create DataFrame and scale
example_df = pd.DataFrame([example_house])
example_scaled = scaler.transform(example_df)

# Get predictions from best model
best_model = models[best_model_name]
predicted_price = best_model.predict(example_scaled)[0]

print(f"Example House Details:")
print(f"Median Income: ${example_house['MedInc']*100}k")
print(f"House Age: {example_house['HouseAge']} years")
print(f"Average Rooms: {example_house['AveRooms']}")
print(f"Predicted Price: ${predicted_price:,.2f}")

## 9. Summary

In [ ]:
print("🏠 HOUSE PRICE PREDICTION SUMMARY")
print("=" * 50)
print(f"Dataset: {len(df)} samples, {len(X.columns)} features")
print(f"Best Model: {best_model_name}")
print(f"RMSE: ${comparison_df.iloc[0]['RMSE']:,.2f}")
print(f"MAE: ${comparison_df.iloc[0]['MAE']:,.2f}")
print(f"R² Score: {comparison_df.iloc[0]['R²']:.4f}")
print("\nTop 3 Most Important Features:")
top_features = feature_importance.tail(3)[::-1]
for idx, row in top_features.iterrows():
    print(f"  {row['feature']}: {row['importance']:.3f}")